# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration and processing of the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, referencing all data elements by their `@id` for precision and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s. We use Croissant metadata objects to discover the dataset structure and the corresponding `@id`s, which are used for all further data access.

In [ ]:
# List all record sets with their @id and name
record_sets = []
print("Available record sets:")
for rs in metadata.record_sets:
    print(f" - @id: {rs.id}, name: {rs.name}")
    record_sets.append(rs.id)

record_set_fields = {}
for rs in metadata.record_sets:
    print(f"\nFields for record set @id: {rs.id}, name: {rs.name}")
    fields = []
    for field in rs.fields:
        print(f"   - Field @id: {field.id}, name: {field.name}, data type: {field.data_type}")
        fields.append(field.id)
    record_set_fields[rs.id] = fields

if len(record_sets):
    sample_rs = record_sets[0]
    print(f"\nSample records from record set {sample_rs} (first 2):")
    for i, row in enumerate(dataset.records(record_set=sample_rs)):
        pprint.pprint(row)
        if i >= 1:
            break

## 3. Data Extraction
Load each record set into a pandas DataFrame for analysis, referencing record sets and fields by their Croissant `@id` values.

In [ ]:
# Extract all data for each primary record set
dataframes = {}
print("\nLoading data from record sets:")
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"- Loaded DataFrame for record set @id: {record_set_id} with {len(df)} rows and columns: {df.columns.tolist()}")
    else:
        print(f"- No data for record set @id: {record_set_id}")

# Example: Show first 5 rows of the first record set (if loaded)
sample_record_set_id = record_sets[0] if record_sets else None
if sample_record_set_id in dataframes:
    print(f"\nColumns in sample DataFrame (@id: {sample_record_set_id}):")
    print(dataframes[sample_record_set_id].columns.tolist())
    dataframes[sample_record_set_id].head()
else:
    print('No available dataframes.')

## 4. Exploratory Data Analysis (EDA)
This section demonstrates how to process and analyze the dataset. Operations include filtering records by a numeric field, normalizing data, and grouping by a categorical field—all using their corresponding Croissant `@id`s.

> *Tip: Always use the `@id` of record sets and fields in data references for clarity and reproducibility.*

In [ ]:
# Pick a record set and numeric field @id for EDA
if len(dataframes):
    # Use the first available record set as example
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Working with DataFrame for record set @id: {record_set_id}")
    numeric_field_id = None

    # Attempt to find a numeric column from metadata
    for rs in metadata.record_sets:
        if rs.id == record_set_id:
            for field in rs.fields:
                if field.data_type in ('schema:Float', 'schema:Number', 'schema:Integer') and field.id in df.columns:
                    numeric_field_id = field.id
                    print(f"Selected numeric field @id: {numeric_field_id} ({field.name}) for EDA")
                    break
            break

    if numeric_field_id and numeric_field_id in df.columns:
        # Filtering
        threshold = 0  # Change as appropriate for the field
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where @{numeric_field_id} > {threshold} (showing 5):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized @{numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping (find first categorical or string field)
        group_field_id = None
        for rs in metadata.record_sets:
            if rs.id == record_set_id:
                for field in rs.fields:
                    if field.data_type == 'schema:Text' and field.id in df.columns:
                        group_field_id = field.id
                        print(f"Grouping by field @id: {group_field_id} ({field.name})")
                        break
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nMean @{numeric_field_id} grouped by @{group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA. Check field data types in metadata.")
else:
    print("No DataFrames loaded for analysis.")

## 5. Visualization
Plotting the distribution of a numeric field and group-wise means using matplotlib (or seaborn if available). All visualizations reference fields by Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution and group means
if len(dataframes) and 'numeric_field_id' in locals() and numeric_field_id:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of @{numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field is available, plot group means
    if 'group_field_id' in locals() and group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean @{numeric_field_id} by @{group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('No data available for plotting. Ensure numeric_field_id is set and DataFrame loaded.')

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset via its Croissant schema, reviewed its structure using `@id`-based referencing, loaded records into pandas DataFrames, performed exploratory data analysis, and visualized distributions. This approach using `mlcroissant` and strict `@id` referencing ensures clear, reproducible, and scalable scientific data workflows.